# 01 - Exploratory Data Analysis (EDA)

Loading and exploring the CIC-DDoS2019 dataset for multi-class classification.

In [1]:
# Imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

sns.set_style("whitegrid")
%matplotlib inline

## Load Data (with sampling)

In [ ]:
# Configuration
DATA_DIR = "../data/raw/CSVs"
SAMPLE_ROWS = 5000  # rows per CSV file
RANDOM_STATE = 42

# Find all CSV files
csv_files = glob.glob(f"{DATA_DIR}/**/*.csv", recursive=True)
print(f"Found {len(csv_files)} CSV files")
for f in csv_files:
    print(f"  - {f}")

Found 18 CSV files
  - ../data/raw/CSVs/03-11/Syn.csv
  - ../data/raw/CSVs/03-11/UDP.csv
  - ../data/raw/CSVs/03-11/NetBIOS.csv
  - ../data/raw/CSVs/03-11/LDAP.csv
  - ../data/raw/CSVs/03-11/MSSQL.csv
  - ../data/raw/CSVs/03-11/Portmap.csv
  - ../data/raw/CSVs/03-11/UDPLag.csv
  - ../data/raw/CSVs/01-12/Syn.csv
  - ../data/raw/CSVs/01-12/TFTP.csv
  - ../data/raw/CSVs/01-12/DrDoS_UDP.csv
  - ../data/raw/CSVs/01-12/DrDoS_DNS.csv
  - ../data/raw/CSVs/01-12/DrDoS_LDAP.csv
  - ../data/raw/CSVs/01-12/DrDoS_SNMP.csv
  - ../data/raw/CSVs/01-12/DrDoS_SSDP.csv
  - ../data/raw/CSVs/01-12/DrDoS_MSSQL.csv
  - ../data/raw/CSVs/01-12/UDPLag.csv
  - ../data/raw/CSVs/01-12/DrDoS_NTP.csv
  - ../data/raw/CSVs/01-12/DrDoS_NetBIOS.csv


In [ ]:
# Load and sample data
def load_and_sample(filepath, n_rows=SAMPLE_ROWS, random_state=RANDOM_STATE):
    """Load CSV with optional sampling."""
    df = pd.read_csv(filepath)
    if len(df) > n_rows:
        return df.sample(n=n_rows, random_state=random_state)
    return df

# Load all files
dfs = []
for f in csv_files:
    print(f"Loading {os.path.basename(f)}...")
    df = load_and_sample(f)
    dfs.append(df)
    print(f"  -> {len(df)} rows")

# Combine
df = pd.concat(dfs, ignore_index=True)
print(f"\nTotal: {len(df):,} rows, {df.shape[1]} columns")

Loading Syn.csv...


/tmp/ipykernel_50954/3037850693.py:4: DtypeWarning: Columns (0: SimillarHTTP) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filepath)


  -> 50000 rows
Loading UDP.csv...
  -> 50000 rows
Loading NetBIOS.csv...
  -> 50000 rows
Loading LDAP.csv...
  -> 50000 rows
Loading MSSQL.csv...
  -> 50000 rows
Loading Portmap.csv...
  -> 50000 rows
Loading UDPLag.csv...
  -> 50000 rows
Loading Syn.csv...
  -> 50000 rows
Loading TFTP.csv...


In [ ]:
# Basic info
print("Columns:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes.value_counts())

## Target Variable Distribution

In [ ]:
# Label distribution
label_counts = df['Label'].value_counts()
print("Label distribution:")
print(label_counts)
print(f"\nUnique labels: {df['Label'].nunique()}")

In [ ]:
# Plot label distribution
plt.figure(figsize=(12, 6))
label_counts.plot(kind='bar')
plt.title('Attack Type Distribution')
plt.xlabel('Label')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('reports/figures/label_distribution.png', dpi=150)
plt.show()

## Key Features: Duration, Packet Count, Header Size

In [ ]:
# Focus features (from CICFlowMeter)
KEY_FEATURES = [
    'Flow Duration',
    'Total Fwd Packets',
    'Total Backward Packets',
    'Fwd Header Length',
    'Bwd Header Length',
    'Protocol',
    'Flow Bytes/s',
    'Flow Packets/s'
]

# Check which exist
available_features = [f for f in KEY_FEATURES if f in df.columns]
print(f"Available key features: {len(available_features)}")
for f in available_features:
    print(f"  - {f}")

In [ ]:
# Summary stats for key features
print(df[available_features].describe())

In [ ]:
# Boxplots by attack type
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

features_to_plot = ['Flow Duration', 'Total Fwd Packets', 'Protocol', 'Flow Bytes/s']
for ax, feat in zip(axes.flatten(), features_to_plot):
    if feat in df.columns:
        df.boxplot(column=feat, by='Label', ax=ax)
        ax.set_title(feat)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

plt.suptitle('')
plt.tight_layout()
plt.savefig('reports/figures/features_boxplot.png', dpi=150)
plt.show()

## Save Processed Data

In [ ]:
# Save sampled data
output_dir = "data/processed"
os.makedirs(output_dir, exist_ok=True)
df.to_csv(f"{output_dir}/ddos_sampled.csv", index=False)
print(f"Saved {len(df):,} rows to {output_dir}/ddos_sampled.csv")